In [1]:
import os
import sys

current_path = os.path.abspath(os.getcwd())
base_path = os.path.abspath(os.path.join(current_path, '../../../../'))
sys.path.append(base_path)

import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

import numpy as np
from tqdm import tqdm
import datetime

from data_loaders.data_loader_baseline_clusters import DataLoaderBaselineClusters
import project_images_loader as pil

import feature_extractor as feature_extractor

import utils.functions as fc
import utils.util_deep_learning as udl
import utils.util as util
import utils.visualization as vis

tf.keras.backend.set_image_data_format("channels_first")
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

# tf.compat.v1.mixed_precision.disable_mixed_precision_graph_rewrite()
print(tf.config.list_physical_devices('GPU'))

2024-04-15 15:27:16.800292: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-04-15 15:27:16.883965: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-04-15 15:27:16.884004: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-04-15 15:27:16.885269: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-04-15 15:27:16.893335: I tensorflow/core/platform/cpu_feature_guar

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA A100 80GB PCIe, compute capability 8.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Visualize data

In [2]:
# data_loader = DataLoaderBaselineClusters(pil.INAME_PREPROCESSED, 
#                                     f"{current_path}/clusters/C16_T96_vol_diff_perc.csv", 
#                                     batch_size=2, 
#                                     image_size=(192, 224, 192),
#                                     channels_first=True,
#                                     data_type=np.float32,
#                                     return_clusters=True, 
#                                     exclude_ids=None, 
#                                     id_column=0, 
#                                     cluster_column=1, 
#                                     onehot_encoder=True, 
#                                     train_fraction=0.8, 
#                                     shuffle_train=True, 
#                                     shuffle_val=False, 
#                                     current_mode='train',
#                                     server=False)

# imgs, clusters = data_loader[0]
# print(imgs.shape, clusters.shape)

# for i in range(5):  
#     imgs, clusters = data_loader[i]
#     fc.imgshow_3D(str(clusters[0, :]), imgs[0, 0, :, :, :])
    

### Loss functions

In [3]:

def loss_CategoricalFocalCrossentropy(data_clusters, gamma=2.0, alpha=1.0):
    classes_weights = data_clusters.get_class_weights(alpha=alpha, mode='train')
    loss_fn = tf.keras.losses.CategoricalFocalCrossentropy(alpha=classes_weights, gamma=gamma)
    return loss_fn

def loss_cross_entropy():
    return tf.keras.losses.CategoricalCrossentropy()

# loss_fn = loss_CategoricalFocalCrossentropy()
# loss_fn = tf.keras.losses.CategoricalCrossentropy()



In [4]:
@tf.function
def val_step(Fe, h3_batch, y, loss_fn):
    y_pred = Fe(h3_batch, training=False)
    loss = loss_fn(y, y_pred)
    return loss, y_pred

@tf.function
def train_step(Fe, h3_batch, y, optimizer, loss_fn):
    with tf.GradientTape() as fe_tape:
        y_pred = Fe(h3_batch, training=True)
        loss = loss_fn(y, y_pred)
        scaled_loss = optimizer.get_scaled_loss(loss)

    scaled_gradients = fe_tape.gradient(scaled_loss, Fe.trainable_variables)
    gradients = optimizer.get_unscaled_gradients(scaled_gradients)
    optimizer.apply_gradients(zip(gradients, Fe.trainable_variables))

    return loss, y_pred


### Training

In [5]:

def train(num_epochs, batch_size):


    # ---- Data
    server = True
    input_shape = (192,224,192)
    remove_patients = ['01388', '06611', '02408', '03139', '02507']

    data_loader = DataLoaderBaselineClusters(pil.INAME_PREPROCESSED, 
                                        f"{current_path}/clusters/bpf_clusters_4.csv", 
                                        batch_size=batch_size, 
                                        image_size=input_shape,
                                        channels_first=True,
                                        data_type=np.float32,
                                        return_clusters=True, 
                                        exclude_ids=remove_patients, 
                                        id_column=0, 
                                        cluster_column=1, 
                                        onehot_encoder=True, 
                                        train_fraction=0.8, 
                                        shuffle_train=True, 
                                        shuffle_val=False, 
                                        current_mode='train',
                                        server=server)

    # ---- Models
    number_classes = data_loader.get_number_of_classes()
    genesis_weights = f"{current_path}/genesis_model/Genesis_Chest_CT.h5"
    Fe = feature_extractor.create_model((1,)+input_shape, number_classes, genesis_weights)
    
    # loass funcitons
    # loss_fn = loss_CategoricalFocalCrossentropy(data_loader, gamma=2.0, alpha=1.0)
    loss_fn = loss_cross_entropy()

    # optimizers
    inital_lr = 1e-1
    optimizer = tf.keras.optimizers.Adam(inital_lr)
    optimizer = mixed_precision.LossScaleOptimizer(optimizer)
    optimizer_monitor = udl.LearningRateMonitor(optimizer, inital_lr, 0.5, 1e-6, 2, "loss")

    # ---- checkpoint
    checkpoint_dir = f"{current_path}/models/training_checkpoints/"
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
    checkpoint = tf.train.Checkpoint(optimizer=optimizer, Fe=Fe)
    # restore checkpoint
    # checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

    # visualization
    nb_loss = 2
    acum_loss_train = udl.AcumulativeMetrics(nb_metrics = nb_loss)
    acum_loss_val = udl.AcumulativeMetrics(nb_metrics = nb_loss)
    acum_classification_train = udl.AccumulativeClassificationMetrics(number_classes)
    acum_classification_val = udl.AccumulativeClassificationMetrics(number_classes)


    log_dir=f"{current_path}/models/logs/train"
    summary_writer = tf.summary.create_file_writer(log_dir)


    for epoch in range(0,num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # ---- Train
        acum_loss_train.reset()
        data_loader.change_mode('train')
        progress_bar = tqdm(total=len(data_loader), desc=f'Initializing...')
        for step, (h3_batch, y) in enumerate(data_loader):
            loss, y_pred = train_step(Fe, h3_batch, y, optimizer, loss_fn)
            y_pred_onehot = tf.one_hot(tf.argmax(y_pred, axis=1), depth=y_pred.shape[1])

            acum_classification_train.update(y, y_pred_onehot)
            acc_mean = acum_classification_train.get_acc()
            acc_text = acum_classification_train.get_progress_bar_text()

            acum_loss_train.update((loss, acc_mean))
            losses_text = acum_loss_train.get_progress_bar_text(['loss_tr', 'acc'])

            progress_bar.set_description(desc=f"{losses_text}\t{acc_text}")
            progress_bar.update(1)

        acum_classification_train.reset()
        del progress_bar

        
        # # ---- val
        acum_loss_val.reset()
        data_loader.change_mode('val')
        progress_bar = tqdm(total=len(data_loader), desc=f'Initializing...')
        for step, (h3_batch, y) in enumerate(data_loader):
            loss, y_pred = val_step(Fe, h3_batch, y, loss_fn)
            y_pred_onehot = tf.one_hot(tf.argmax(y_pred, axis=1), depth=y_pred.shape[1])
  
            acum_classification_val.update(y, y_pred_onehot)
            acc_mean = acum_classification_val.get_acc()
            acc_text = acum_classification_val.get_progress_bar_text()
        
            acum_loss_val.update((loss, acc_mean))
            losses_text = acum_loss_val.get_progress_bar_text(['loss_vl', 'acc'])

            progress_bar.set_description(desc=f"{losses_text}\t{acc_text}")
            progress_bar.update(1)

        acum_classification_val.reset()
        del progress_bar


        # ---- Check points
        # if  epoch > 0 and epoch % 2 == 0:
        checkpoint.save(file_prefix=checkpoint_prefix)
        
        # ---- Learning rate
        loss_mean_val = acum_loss_val.get_mean()
        optimizer_monitor.on_epoch_end(loss_mean_val[0])


        # ---- tensorboard
        loss_mean_train = acum_loss_train.get_mean()
        loss_mean_val = acum_loss_train.get_mean()
        with summary_writer.as_default():
            tf.summary.scalar('loss_train', loss_mean_train[0], step=epoch)
            tf.summary.scalar('loss_val', loss_mean_val[0], step=epoch)

            tf.summary.scalar('acc_train', loss_mean_train[1], step=epoch)
            tf.summary.scalar('acc_val', loss_mean_val[1], step=epoch)


train(num_epochs=50, batch_size=1)
# 7.95463

 ------------------- Data Loader ------------------- 
Total: 698, Per clusters: [149 205 207 137]
Nb images Train: 560, Val: 138
Nb images per cluster Train: [120 164 166 110], Val: [29 41 41 27]


2024-04-15 15:28:10.034847: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79086 MB memory:  -> device: 0, name: NVIDIA A100 80GB PCIe, pci bus id: 0000:c2:00.0, compute capability: 8.0



Epoch 1/50


Initializing...:   0%|          | 0/560 [00:00<?, ?it/s]2024-04-15 15:28:15.214101: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-04-15 15:28:15.342843: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-04-15 15:28:15.585671: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2024-04-15 15:28:36.944303: I external/local_xla/xla/service/service.cc:168] XLA service 0x7eeefd3659d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-04-15 15:28:36.944335: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100 80GB PCIe, Compute Capability 8.0
2024-04-15 15:28:36.952691: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.



Epoch 2/50


...

loss_tr: 1.37150	acc: 0.33924	[0/120, 403/164, 157/166, 0/110]: 100%|██████████| 560/560 [03:14<00:00,  2.88it/s]
loss_vl: 1.36993	acc: 0.12900	[0/29, 0/41, 138/41, 0/27]: 100%|██████████| 138/138 [00:38<00:00,  3.57it/s]



Epoch 16/50


loss_tr: 1.37556	acc: 0.26568	[0/120, 118/164, 442/166, 0/110]: 100%|██████████| 560/560 [03:23<00:00,  2.75it/s]
loss_vl: 1.36849	acc: 0.12900	[0/29, 0/41, 138/41, 0/27]: 100%|██████████| 138/138 [00:32<00:00,  4.29it/s]



Epoch 17/50


loss_tr: 1.37828	acc: 0.24574	[0/102, 167/124, 271/128, 0/84]:  78%|███████▊  | 438/560 [02:31<00:41,  2.95it/s]

In [ ]:
# ---- Data loader
input_shape = (192,224,192)
batch_size = 1
server = True
remove_patients = ['01388', '06611', '02408', '03139', '02507']
data_loader = DataLoaderBaselineClusters(pil.INAME_PREPROCESSED, 
                                    f"{current_path}/clusters/C16_T96_vol_diff_perc.csv", 
                                    batch_size=batch_size, 
                                    image_size=input_shape,
                                    channels_first=True,
                                    data_type=np.float32,
                                    return_clusters=True, 
                                    exclude_ids=remove_patients, 
                                    id_column=0, 
                                    cluster_column=1, 
                                    onehot_encoder=True, 
                                    train_fraction=0.8, 
                                    shuffle_train=True, 
                                    shuffle_val=False, 
                                    current_mode='train',
                                    server=server)

# ---- Model
number_classes = data_loader.get_number_of_classes()
Fe = feature_extractor.create_model((1,)+input_shape, number_classes, None)
checkpoint_dir = f"{current_path}/models/training_checkpoints/"
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt")
checkpoint = tf.train.Checkpoint(Fe=Fe)
# checkpoint.restore(tf.train.latest_checkpoint(checkpoint_dir))

# ---- Loss
loss_fn = loss_CategoricalFocalCrossentropy(data_loader, gamma=2.0, alpha=1.0)
nb_loss =2
acum_loss_val = udl.AcumulativeMetrics(nb_metrics = nb_loss)
accuracy_val = tf.keras.metrics.Accuracy()

# # ---- val
data_loader.change_mode('val')
for step, (h3_batch, y) in enumerate(data_loader):
    loss, y_pred = val_step(Fe, h3_batch, y, loss_fn)
    y_pred_onehot = tf.one_hot(tf.argmax(y_pred, axis=1), depth=y_pred.shape[1])
    accuracy_val.update_state(y, y_pred_onehot)
    acc = accuracy_val.result().numpy()

    print(y)
    print(y_pred_onehot.numpy())
    print(y_pred)
    print(f"loss: {loss}, acc: {acc}")


 ------------------- Data Loader ------------------- 
Total: 303, Per clusters: [35 10 33 19 19 25 28 19 29  3  5 21 24  9  9 15]
Nb images Train: 249, Val: 54
Nb images per cluster Train: [28  8 27 16 16 20 23 16 24  2  4 17 20  8  8 12], Val: [7 2 6 3 3 5 5 3 5 1 1 4 4 1 1 3]
[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
tf.Tensor(
[[0.06254512 0.06265762 0.06237437 0.06239614 0.06241406 0.06237371
  0.06266575 0.06258697 0.06254822 0.06245047 0.06255046 0.06250887
  0.06244436 0.06256764 0.06244297 0.06247331]], shape=(1, 16), dtype=float32)
loss: 0.08699917793273926, acc: 0.875
[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
[[0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
tf.Tensor(
[[0.06255143 0.06267982 0.0623556  0.06238146 0.06240122 0.06235691
  0.06268843 0.06260036 0.06255519 0.06244367 0.06255609 0.06250989
  0.06243745 0.06257727 0.06243465 0.06247053]], shape=(1, 16), dtype=float32)
loss: 0.086994834244251

In [ ]:
# Epoch 11/50
# loss_vl: 1.01357	acc: 0.92224: 100%|██████████| 27/27 [00:09<00:00,  2.73it/s]
# loss_tr: 0.97031	acc: 0.89078: 100%|██████████| 125/125 [01:19<00:00,  1.58it/s]